# DX 704 Week 4 Project

This week's project will test the learning speed of linear contextual bandits compared to unoptimized approaches.
You will start with building a preference data set for evaluation, and then implement different variations of LinUCB and visualize how fast they learn the preferences.


The full project description, a template notebook and supporting code are available on GitHub: [Project 4 Materials](https://github.com/bu-cds-dx704/dx704-project-04).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

In [2]:
# =====================
# Important Libraries
# =====================

%pip install numpy pandas scikit-learn contextualbandits

import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

import contextualbandits as cb

Note: you may need to restart the kernel to use updated packages.


## Part 1: Collect Rating Data

The file "recipes.tsv" in this repository has information about 100 recipes.
Make a new file "ratings.tsv" with two columns, recipe_slug (from recipes.tsv) and rating.
Populate the rating column with values between 0 and 1 where 0 is the worst and 1 is the best.
You can assign these ratings however you want within that range, but try to make it reflect a consistent set of preferences.
These could be your preferences, or a persona of your choosing (e.g. chocolate lover, bacon-obsessed, or sweet tooth).
Make sure that there are at least 10 ratings of zero and at least 10 ratings of one.


Hint: You may find it more convenient to assign raw ratings from 1 to 5 and then remap them as follows.

`ratings["rating"] = (ratings["rating_raw"] - 1) * 0.25`

In [3]:
# ======================
# Inspect Recipes Data
# ======================
recipes_df = pd.read_csv('recipes.tsv', sep='\t')
# print(recipes_df.head(10))


# ========================
# Populate Rating Column 
# ========================
# use values between 0 and 1, 10 ratings of 0, and 10 ratings of 1
ratings = np.random.choice([0.25, 0.5, 0.75], size=len(recipes_df))
ratings[:10] = 1
ratings[10:20] = 0
np.random.shuffle(ratings)

# Create ratings df using recipe_slug and ratings
ratings_df = pd.DataFrame({
    'recipe_slug': recipes_df['recipe_slug'],
    'rating': ratings
    })

# print(ratings_df.head(10))


# ==============================================
# Save results to TSV file named 'ratings.tsv'
# ==============================================
ratings_df.to_csv('ratings.tsv', sep='\t', index=False)
ratings_df

,recipe_slug,rating
0,falafel,1.00
1,spamburger,0.75
2,bacon-fried-rice,0.75
3,chicken-fingers,0.25
4,apple-crisp,0.75
...,...,...
95,bacon-mac-and-cheese,0.00
96,chicken-alfredo-lasagna,1.00
97,classic-beef-lasagna,0.00
98,vegetarian-mushroom-lasagna,0.50


Submit "ratings.tsv" in Gradescope.

## Part 2: Construct Model Input

Use your file "ratings.tsv" combined with "recipe-tags.tsv" to create a new file "features.tsv" with a column recipe_slug, a column bias which is hard-coded to one, and a column for each tag that appears in "recipe-tags.tsv".
The tag column in this file should be a 0-1 encoding of the recipe tags for each recipe.
[Pandas reshaping function methods](https://pandas.pydata.org/docs/user_guide/reshaping.html) may be helpful.

The bias column will make later LinUCB calculations easier since it will just be another dimension.

Hint: For later modeling steps, it will be important to have the feature data (inputs) and the rating data (target outputs) in the same order.
It is highly recommended to make sure that "features.tsv" and "ratings.tsv" have the recipe slugs in the same order.

In [4]:
# ======================
# Construct Model Input
# ======================
recipe_tags_df = pd.read_csv('recipe-tags.tsv', sep='\t')

# Create bias column 
tag_features_df = (
    pd.crosstab(recipe_tags_df['recipe_slug'], recipe_tags_df['recipe_tag'])
    .clip(upper=1)
    .reindex(ratings_df['recipe_slug'], fill_value=0)
    .reset_index(drop=True)
    )

# Create features df using recipe_slug and bias column
features_df = pd.DataFrame({
    'recipe_slug': ratings_df['recipe_slug'].to_numpy(),
    'bias': 1,
    })

features_df = pd.concat([features_df, tag_features_df], axis=1)

# print(features_df.head(10))


# ==============================================
# Save results to TSV file named 'features.tsv'
# ==============================================
features_df.to_csv('features.tsv', sep='\t', index=False)
features_df

,recipe_slug,bias,alfredo,almond,american,appetizer,appetizers,apple,asiancuisine,asparagus,...,udonnoodles,vanilla,vanillaicecream,vegan,vegetables,vegetarian,warm,whippedcream,winter,yeastdough
0,falafel,1,0,0,0,1,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,spamburger,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,bacon-fried-rice,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,chicken-fingers,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,apple-crisp,1,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,bacon-mac-and-cheese,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
96,chicken-alfredo-lasagna,1,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
97,classic-beef-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
98,vegetarian-mushroom-lasagna,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


Submit "features.tsv" in Gradescope.

## Part 3: Linear Preference Model

Use your feature and rating files to build a ridge regression model with ridge regression's regularization parameter $\alpha$ set to 1.


Hint: If you are using scikit-learn modeling classes, you should use `fit_intercept=False` since that intercept value will be redundant with the bias coefficient.

Hint: The estimate component of the bounds should match the previous estimate, so you should be able to just focus on the variance component of the bounds now.

In [5]:
# =========================
# Ridge Regression Model
# =========================
# use feature and rating files to build ridge regression model
features_df = pd.read_csv('features.tsv', sep='\t')
ratings_df = pd.read_csv('ratings.tsv', sep='\t')

# Select features and target
X_all = features_df.drop(columns=['recipe_slug'])
y = ratings_df['rating'].values
feature_names = [c for c in features_df.columns if c != 'recipe_slug']

# Standardize features
# scaler = StandardScaler(with_mean=True, with_std=True)
# X_scaled = scaler.fit_transform(X_all)

# "Lambda" in many texts corresponds to alpha in sklearn's Ridge
lambda_param = 1.0

# Initialize and fit Ridge regression model
ridge_model = Ridge(alpha=lambda_param, fit_intercept=False)
ridge_model.fit(X_all, y)

# Results
print(f'Ridge coefficients with lambda = {lambda_param}:')
for name, coef in zip(feature_names, ridge_model.coef_): 
    if name in {"x3", "x8"}:
        print(f"*** {name}: {coef: .6f}  <-- truly nonzero in data-generating process")
    else:
        print(f"    {name}: {coef: .6f}")

print(f'Ridge Regression Intercept: {ridge_model.intercept_}')

Ridge coefficients with lambda = 1.0:
    bias:  0.304819
    alfredo:  0.128600
    almond:  0.082083
    american: -0.109488
    appetizer:  0.044954
    appetizers:  0.136771
    apple:  0.088148
    asiancuisine: -0.026728
    asparagus:  0.068263
    avocado: -0.074982
    bacon: -0.039504
    baked:  0.056835
    bakeddishes: -0.058919
    bakery: -0.078502
    baking:  0.227104
    beans: -0.024257
    beef: -0.072501
    bellpeppers:  0.058866
    berries: -0.153680
    blackbeans:  0.058866
    blackvinegar: -0.039964
    blueberry: -0.017628
    boiledeggs: -0.003149
    bokchoy: -0.003149
    braided: -0.044333
    bread: -0.067068
    breaded:  0.094402
    breakfast:  0.214569
    breakfastpastry:  0.025305
    brioche: -0.017926
    broiling: -0.014439
    brownies: -0.014817
    brownsugar: -0.047144
    brunch:  0.043582
    burger: -0.026177
    burritos: -0.006915
    buttery: -0.105465
    cake: -0.021596
    carrots: -0.003149
    casserole: -0.041895
    celebratio

Save the coefficients of this model in a file "model.tsv" with columns "recipe_tag" and "coefficient".
Do not add anything for the `intercept_` attribute of a scikit-learn model; this will be covered by the coefficient for the bias column added in part 2.

In [6]:
# ================================================
# Save coefficients to TSV file named 'model.tsv'
# ================================================
model_df = pd.DataFrame({ 
    'recipe_tag': [c for c in feature_names if c != 'bias'],        
    'coefficient': [coef for i, coef in enumerate(ridge_model.coef_) if feature_names[i] != 'bias']    
})

# print(model_df.head(10))
 
# Save file
model_df.to_csv('model.tsv', sep='\t', index=False)
model_df

,recipe_tag,coefficient
0,alfredo,0.128600
1,almond,0.082083
2,american,-0.109488
3,appetizer,0.044954
4,appetizers,0.136771
...,...,...
291,vegetarian,0.096592
292,warm,0.025305
293,whippedcream,0.022447
294,winter,0.023837


Submit "model.tsv" in Gradescope.

## Part 4: Recipe Estimates

Use the recipe model to estimate the score of every recipe.
Save these estimates to a file "estimates.tsv" with columns recipe_slug and score_estimate.

In [7]:
# ============================================
# Use Model to Estimate Score of Each Recipe
# ============================================
recipe_scores = ridge_model.predict(X_all)
recipe_scores


# ================================================
# Save coefficients to TSV file named 'model.tsv'
# ================================================
estimates_df = pd.DataFrame({
    'recipe_slug': features_df['recipe_slug'],
    'score_estimate': recipe_scores
    })

# print(estimates_df.head(10))

# Save file
estimates_df.to_csv('estimates.tsv', sep='\t', index=False)
estimates_df

,recipe_slug,score_estimate
0,falafel,0.888574
1,spamburger,0.694954
2,bacon-fried-rice,0.699667
3,chicken-fingers,0.337785
4,apple-crisp,0.656572
...,...,...
95,bacon-mac-and-cheese,0.056390
96,chicken-alfredo-lasagna,0.871400
97,classic-beef-lasagna,0.100766
98,vegetarian-mushroom-lasagna,0.491951


Submit "estimates.tsv" in Gradescope.

## Part 5: LinUCB Bounds

Calculate the upper bounds of LinUCB using data corresponding to trying every recipe once and receiving the rating in "ratings.tsv" as the reward.
Keep the ridge regression regularization parameter at 1, and set LinUCB's $\alpha$ parameter to 2.
Save these upper bounds to a file "bounds.tsv" with columns recipe_slug and score_bound.

In [8]:
# ==========================
# Calculate LinUCB Bounds
# ==========================
alpha_linucb = 2.0

X_np = X_all.to_numpy()
A = X_np.T @ X_np + lambda_param * np.eye(X_all.shape[1])
A_inv = np.linalg.inv(A)

variance = np.einsum('ij,jk,ik->i', X_np, A_inv, X_np)  # x^T A^-1 x per recipe

score_bounds = recipe_scores + alpha_linucb * np.sqrt(variance)


# ================================================
# Save upper bounds to TSV file named 'bounds.tsv'
# ================================================
bounds_df = pd.DataFrame({
    'recipe_slug': features_df['recipe_slug'],
    'score_bound': score_bounds
    })

# print(bounds_df.head(10))

# Save file
bounds_df.to_csv('bounds.tsv', sep='\t', index=False)
bounds_df

,recipe_slug,score_bound
0,falafel,2.654829
1,spamburger,2.589290
2,bacon-fried-rice,2.565824
3,chicken-fingers,2.131900
4,apple-crisp,2.432914
...,...,...
95,bacon-mac-and-cheese,1.877944
96,chicken-alfredo-lasagna,2.639473
97,classic-beef-lasagna,1.811259
98,vegetarian-mushroom-lasagna,2.082071


Submit "bounds.tsv" in Gradescope.

## Part 6: Make Online Recommendations

Implement LinUCB to make 100 recommendations starting with no data and using the same parameters as in part 5.
One recommendation should be made at a time and you can break ties arbitrarily.
After each recommendation, use the rating from part 1 as the reward to update the LinUCB data.
Record the recommendations made in a file "recommendations.tsv" with columns "recipe_slug", "score_bound", and "reward".
The rows in this file should be in the same order as the recommendations were made.

Hint: do not remove recipes after each recommendation.
Repeating recommendations is expected.

In [9]:
# =============================
# Make Online Recommendations
# =============================
# Implement LinUCB to make 100 recommendations
n_features = X_np.shape[1]
recipe_slugs = features_df['recipe_slug'].to_numpy()
rewards = y  # ratings in the same order as X_np

# Start with no data
A_online = lambda_param * np.eye(n_features)
b_online = np.zeros(n_features)

recommendations = []

for t in range(100):
    A_inv_online = np.linalg.inv(A_online)
    theta = A_inv_online @ b_online

    estimates = X_np @ theta
    variance_online = np.einsum('ij,jk,ik->i', X_np, A_inv_online, X_np)
    score_bounds_online = estimates + alpha_linucb * np.sqrt(variance_online)

    chosen = np.argmax(score_bounds_online)  # ties broken arbitrarily (first max)

    x = X_np[chosen]
    reward = rewards[chosen]

    # Update LinUCB data with reward
    A_online += np.outer(x, x)
    b_online += reward * x

    recommendations.append({
        'recipe_slug': recipe_slugs[chosen],
        'score_bound': score_bounds_online[chosen],
        'reward': reward
        })


# =============================================================
# Save recommendations to TSV file named 'recommendations.tsv'
# =============================================================
recommendations_df = pd.DataFrame(recommendations)

recommendations_df.to_csv('recommendations.tsv', sep='\t', index=False)
recommendations_df

,recipe_slug,score_bound,reward
0,apple-crumble,7.483315,0.50
1,ma-la-chicken,7.225922,1.00
2,quesadillas,7.285068,1.00
3,ramen,7.291871,0.50
4,spamburger,7.032909,0.75
...,...,...,...
95,spinach-and-ricotta-stuffed-shells,3.075148,0.25
96,chocolate-souffle,2.896252,0.75
97,blueberry-crisp,2.892355,0.75
98,quesadillas,2.810042,1.00


Submit "recommendations.tsv" in Gradescope.

## Part 7: Acknowledgments

Make a file "acknowledgments.txt" documenting any outside sources or help on this project.
If you discussed this assignment with anyone, please acknowledge them here.
If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for.
If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy.
If no acknowledgements are appropriate, just write none in the file.


Submit "acknowledgments.txt" in Gradescope.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.


Submit "project.ipynb" in Gradescope.